# Experiment 5: Enrolling a New Participant

Enrollment is share repair where the target is a new participant who never had
a share. The same repair protocol gives the newcomer a valid share of the
existing group secret, without changing the group public key.


In [ ]:
import sys
sys.path.insert(0, "../src")

from frost import Participant, Aggregator, Scalar, Point, G, Q
from frost.tagged_hash import tagged_hash
from frost.keygen import derive_public_verification_share

# DKG setup (2-of-3)
threshold = 2
n = 3
participants = [Participant(index=i, threshold=threshold, participants=n) for i in range(1, n + 1)]

for p in participants:
    p.init_keygen()
for p in participants:
    p.generate_shares()
for receiver in participants:
    other_shares = tuple(
        sender.shares[receiver.index - 1]
        for sender in participants if sender.index != receiver.index
    )
    receiver.aggregate_shares(other_shares)
for p in participants:
    other_commitments = tuple(
        other.coefficient_commitments[0]
        for other in participants if other.index != p.index
    )
    p.derive_public_key(other_commitments)
for p in participants:
    other_ccs = tuple(
        other.coefficient_commitments
        for other in participants if other.index != p.index
    )
    p.derive_group_commitments(other_ccs)

public_key = participants[0].public_key
print(f"Original group: {threshold}-of-{n}")
print(f"Public key x = {public_key.x}")


## Create Participant 4

P4 is a new member who needs a valid share of the existing group polynomial.
The repair protocol with target_index=4 gives P4 the value f(4).


In [ ]:
# Create new participant (threshold and total don't change for the repair mechanism)
new_p = Participant(index=4, threshold=threshold, participants=n + 1)

# Helpers P1 and P2 enroll P4 (same as repair, but target never had a share)
participants[0].generate_repair_shares((2,), 4)
participants[1].generate_repair_shares((1,), 4)

# Exchange
p1_from_p2 = participants[1].get_repair_share(1)
p2_from_p1 = participants[0].get_repair_share(2)

# Aggregate
participants[0].aggregate_repair_shares((p1_from_p2,))
participants[1].aggregate_repair_shares((p2_from_p1,))

# P4 receives aggregate repair shares and reconstructs
new_p.repair_share((
    participants[0].aggregate_repair_share,
    participants[1].aggregate_repair_share,
))

print(f"P4 enrolled with aggregate share: {new_p.aggregate_share}")


## Verify: Group Public Key Unchanged

Enrollment doesn't change the group polynomial. P4 gets f(4) where f is the
same polynomial from DKG. So the group key f(0)·G is unchanged.


In [ ]:
# Give P4 the group public key and commitments
new_p.public_key = public_key
new_p.group_commitments = participants[0].group_commitments

# Verify: P4's verification share is consistent
expected_Y4 = derive_public_verification_share(participants[0].group_commitments, 4)
actual_Y4 = int(new_p.aggregate_share) * G
print(f"P4 verification share matches group commitments: {expected_Y4 == actual_Y4}")
print(f"Group public key unchanged: {new_p.public_key == public_key}")


## Sign with the New Participant

P1 and P4 (original + new) sign a message together.


In [ ]:
message = b"Welcome P4!"
signers = [participants[0], new_p]
signer_indexes = tuple(p.index for p in signers)
n_total = 4  # now 4 participants

for signer in signers:
    signer.generate_nonce_pair()

all_nonce_pairs = [(Point(), Point())] * n_total
for signer in signers:
    all_nonce_pairs[signer.index - 1] = signer.nonce_commitment_pair
nonce_commitment_pairs = tuple(all_nonce_pairs)

shares = []
for signer in signers:
    z_i = signer.sign(message, nonce_commitment_pairs, signer_indexes)
    shares.append(z_i)

agg = Aggregator(
    public_key, message, nonce_commitment_pairs, signer_indexes,
)
sig_hex = agg.signature(tuple(shares))

# BIP340 verification (must use even-y public key per lift_x convention)
sig_bytes = bytes.fromhex(sig_hex)
R_x = int.from_bytes(sig_bytes[:32], "big")
s = int.from_bytes(sig_bytes[32:], "big")
R = Point.lift_x(R_x)
pk = Point.lift_x(public_key.x)
e_bytes = tagged_hash("BIP0340/challenge", sig_bytes[:32] + pk.to_bytes_xonly() + message)
e = int.from_bytes(e_bytes, "big") % Q
valid = s * G == R + (e * pk)
print(f"Signature with P1 + P4 valid: {valid}")


## What About Disenrollment?

You cannot cryptographically *remove* a participant. Once P4 has f(4), they
hold a valid share forever. The group can only:

1. **Refuse to include them** in signing subsets (social, not cryptographic).
2. **Key refresh**: run a new DKG to generate a fresh group secret. The old
   shares (including P4's) become useless for the new key.

This asymmetry, easy enrollment but no cryptographic disenrollment, is
fundamental to polynomial secret sharing.

## Results

Enrollment is share repair targeting a new index. The group polynomial, the
group public key, and all existing shares are unchanged. The new participant
can immediately sign.

## Next Steps

- This experiment verifies the same invariant as
  `test_enrollment.py::test_enrollment`.
- What if you enroll multiple participants? Each enrollment is independent:
  repair targeting index 5, then index 6, etc.
- See Guide Chapter 8 (Advanced Operations) for enrollment details.
